# Transition Matrix Methods for Serial Unemployment (4-State Markov)

**MC vs TM comparison for a consumption-saving model with serially correlated unemployment**

This notebook extends the [2-state Markov prototype](markov-tm-prototype.ipynb) to a
more complex 4-state model drawn from the HARK `MarkovConsumerType` examples.

The four states combine employment status with macroeconomic conditions:

| State | Employment | Economy |
|-------|-----------|----------|
| 0     | Employed  | Boom     |
| 1     | Unemployed| Boom     |
| 2     | Employed  | Bust     |
| 3     | Unemployed| Bust     |

Key features:
- **Degenerate income distributions:** employed agents get $\theta = 1$, unemployed get $\theta = 0$ (zero income).
- **No idiosyncratic uncertainty given the state:** the only randomness comes from
  Markov state transitions.
- **PermGroFac = 1.0** in all states, so a 1D grid over $m$ suffices.

---

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse.linalg as sp_linalg
from copy import copy

from HARK.ConsumptionSaving.ConsMarkovModel import (
    MarkovConsumerType,
    init_indshk_markov,
)
from HARK.distributions import DiscreteDistributionLabeled
from HARK.utilities import make_grid_exp_mult, jump_to_grid_1D

## 1. Model Setup

We construct the 4×4 Markov transition matrix from economic primitives:
average unemployment spell length, unemployment rates in boom/bust, and
transition probabilities between macro states.

**HARK convention:** `MrkvArray` is row-stochastic — `MrkvArray[i, j]`
= P(transition to state $j$ | currently in state $i$).

In [ ]:
unemp_length = 5  # Average length of unemployment spell
urate_good = 0.05  # Unemployment rate in boom
urate_bad = 0.12  # Unemployment rate in bust
bust_prob = 0.01  # P(boom -> bust) per period
recession_length = 20  # Average length of bust

p_reemploy = 1.0 / unemp_length
p_unemploy_good = p_reemploy * urate_good / (1 - urate_good)
p_unemploy_bad = p_reemploy * urate_bad / (1 - urate_bad)
boom_prob = 1.0 / recession_length

# Row-stochastic: MrkvArray[i, j] = P(go to j | in i)
# States: 0=emp-boom, 1=unemp-boom, 2=emp-bust, 3=unemp-bust
MrkvArray = np.array(
    [
        [
            (1 - p_unemploy_good) * (1 - bust_prob),
            p_unemploy_good * (1 - bust_prob),
            (1 - p_unemploy_good) * bust_prob,
            p_unemploy_good * bust_prob,
        ],
        [
            p_reemploy * (1 - bust_prob),
            (1 - p_reemploy) * (1 - bust_prob),
            p_reemploy * bust_prob,
            (1 - p_reemploy) * bust_prob,
        ],
        [
            (1 - p_unemploy_bad) * boom_prob,
            p_unemploy_bad * boom_prob,
            (1 - p_unemploy_bad) * (1 - boom_prob),
            p_unemploy_bad * (1 - boom_prob),
        ],
        [
            p_reemploy * boom_prob,
            (1 - p_reemploy) * boom_prob,
            p_reemploy * (1 - boom_prob),
            (1 - p_reemploy) * (1 - boom_prob),
        ],
    ]
)

J = 4
state_names = ["Emp-Boom", "Unemp-Boom", "Emp-Bust", "Unemp-Bust"]

print("Markov transition matrix (row-stochastic):")
print(np.array2string(MrkvArray, precision=4, suppress_small=True))
print(f"Row sums: {MrkvArray.sum(axis=1)}")

# Stationary distribution
eigvals, eigvecs = np.linalg.eig(MrkvArray.T)
idx = np.argmin(np.abs(eigvals - 1.0))
markov_stationary = eigvecs[:, idx].real
markov_stationary = markov_stationary / markov_stationary.sum()
print("\nStationary distribution:")
for j in range(J):
    print(f"  {state_names[j]:15s}: {markov_stationary[j]:.4f}")

## 2. Solve the Model

In [ ]:
# Start from default Markov params, then customize
params = copy(init_indshk_markov)
params["cycles"] = 0  # infinite horizon
params["AgentCount"] = 100000
params["T_sim"] = 1200
params["Rfree"] = [np.array([1.03, 1.03, 1.03, 1.03])]
params["LivPrb"] = [np.array([0.98, 0.98, 0.98, 0.98])]
params["PermGroFac"] = [np.array([1.0, 1.0, 1.0, 1.0])]  # 1D grid
params["MrkvPrbsInit"] = markov_stationary
params["Mrkv_p11"] = [0.5]  # placeholder — will be overridden
params["Mrkv_p22"] = [0.5]
params["UnempPrb"] = np.zeros(2)  # no extra unemployment on top of Markov
params["global_markov"] = False

agent = MarkovConsumerType(**params)

# Override MrkvArray with our 4×4 matrix AFTER construction
agent.assign_parameters(MrkvArray=[MrkvArray])

# Override income distributions: degenerate (deterministic given state)
employed_income = DiscreteDistributionLabeled(
    pmv=np.ones(1),
    atoms=np.array([[1.0], [1.0]]),
    var_names=["PermShk", "TranShk"],
)
unemployed_income = DiscreteDistributionLabeled(
    pmv=np.ones(1),
    atoms=np.array([[1.0], [0.0]]),
    var_names=["PermShk", "TranShk"],
)
agent.IncShkDstn = [
    [
        employed_income,  # state 0: employed-boom
        unemployed_income,  # state 1: unemployed-boom
        employed_income,  # state 2: employed-bust
        unemployed_income,  # state 3: unemployed-bust
    ]
]

# Re-build terminal solution for 4 states (constructor made it for 2)
from HARK.ConsumptionSaving.ConsMarkovModel import make_markov_solution_terminal

agent.solution_terminal = make_markov_solution_terminal(agent.CRRA, agent.MrkvArray)

agent.solve()
print(f"Solved: {len(agent.solution[0].cFunc)} consumption functions")

In [ ]:
m_plot = np.linspace(0.001, 20, 300)

plt.figure(figsize=(12, 7))
colors = ["#2ecc71", "#e74c3c", "#27ae60", "#c0392b"]
styles = ["-", "-", "--", "--"]
for j in range(J):
    c_vals = agent.solution[0].cFunc[j](m_plot)
    plt.plot(
        m_plot,
        c_vals,
        label=state_names[j],
        linewidth=2,
        color=colors[j],
        linestyle=styles[j],
    )
plt.plot(m_plot, m_plot, ":", color="gray", alpha=0.4, label="45° line")
plt.xlabel("Market resources $m$", fontsize=12)
plt.ylabel("Consumption $c$", fontsize=12)
plt.title(
    "Consumption Functions by Markov State (4-State Serial Unemployment)", fontsize=13
)
plt.legend(fontsize=11)
plt.xlim([0, 20])
plt.ylim([0, 10])
plt.show()

## 3. Monte Carlo Simulation

In [ ]:
agent.track_vars = ["aNrm", "cNrm", "mNrm", "pLvl", "Mrkv"]
agent.initialize_sim()
agent.simulate()

MC_C = np.mean(agent.state_now["mNrm"] - agent.state_now["aNrm"])
MC_A = np.mean(agent.state_now["aNrm"])

print(f"MC Aggregate Consumption = {MC_C:.6f}")
print(f"MC Aggregate Assets      = {MC_A:.6f}")
print()
for j in range(J):
    frac = np.mean(agent.shocks["Mrkv"] == j)
    print(
        f"{state_names[j]:15s}: MC frac = {frac:.4f}, "
        f"stationary = {markov_stationary[j]:.4f}"
    )

In [ ]:
burn_in = 400
mc_aLvls = np.array([np.mean(agent.history["aNrm"][t]) for t in range(agent.T_sim)])

## 4. Transition Matrix Construction

The joint state is $(m, j)$ with $j \in \{0,1,2,3\}$ and $m$ on a
grid of $M$ points.  Total TM size: $(4M)^2$.

Since income distributions are degenerate (1 shock point per state),
each source grid point maps to at most 2 target grid points via the lottery.
This makes the TM very sparse.

In [ ]:
mMin = 0.001
mMax = 50
mCount = 300
mFac = 3

dist_mGrid = make_grid_exp_mult(ming=mMin, maxg=mMax, ng=mCount, timestonest=mFac)

M = len(dist_mGrid)
N_states = M * J

print(f"m-grid: {M} points from {dist_mGrid[0]:.4f} to {dist_mGrid[-1]:.1f}")
print(f"Markov states: {J}")
print(f"Total TM states: {N_states}")

In [ ]:
start = time.time()

MrkvArr = agent.MrkvArray[0]
Rfree_arr = agent.Rfree[0]
LivPrb_arr = agent.LivPrb[0]
PermGroFac_arr = agent.PermGroFac[0]
IncShkDstn_list = agent.IncShkDstn[0]

# Policy on grid
cPol = []
aPol = []
for j in range(J):
    c_j = agent.solution[0].cFunc[j](dist_mGrid)
    a_j = np.maximum(dist_mGrid - c_j, 0.0)
    cPol.append(c_j)
    aPol.append(a_j)

# Newborn distribution
MrkvPrbsInit = markov_stationary
NewBornDist = np.zeros(N_states)
for jp in range(J):
    shk_dstn = IncShkDstn_list[jp]
    newborn_m = jump_to_grid_1D(shk_dstn.atoms[1], shk_dstn.pmv, dist_mGrid)
    NewBornDist[jp * M : (jp + 1) * M] = MrkvPrbsInit[jp] * newborn_m

# Build transition matrix
# MrkvArr[j, jp] = P(j -> jp) [row-stochastic]
TranMatrix = np.zeros((N_states, N_states))

for j in range(J):
    a_grid = aPol[j]
    LivPrb_j = LivPrb_arr[j]

    for jp in range(J):
        markov_prob = MrkvArr[j, jp]
        if markov_prob < 1e-15:
            continue

        Rfree_jp = Rfree_arr[jp]
        PermGroFac_jp = PermGroFac_arr[jp]
        shk_dstn = IncShkDstn_list[jp]
        shk_prbs = shk_dstn.pmv
        perm_shks = shk_dstn.atoms[0]
        tran_shks = shk_dstn.atoms[1]
        bNext = Rfree_jp * a_grid

        for i in range(M):
            mNext = bNext[i] / (perm_shks * PermGroFac_jp) + tran_shks
            lottery_weights = jump_to_grid_1D(mNext, shk_prbs, dist_mGrid)
            src_idx = j * M + i
            TranMatrix[jp * M : (jp + 1) * M, src_idx] += (
                markov_prob * LivPrb_j * lottery_weights
            )

    for i in range(M):
        src_idx = j * M + i
        TranMatrix[:, src_idx] += (1.0 - LivPrb_j) * NewBornDist

elapsed = time.time() - start
print(f"Transition matrix built in {elapsed:.2f} seconds")
print(f"Shape: {TranMatrix.shape}")
col_sums = TranMatrix.sum(axis=0)
print(f"Column sums: min={col_sums.min():.8f}, max={col_sums.max():.8f}")

# Sparsity check (degenerate income => very sparse)
nnz = np.count_nonzero(TranMatrix)
total = N_states * N_states
print(f"Non-zero entries: {nnz} / {total} ({100 * nnz / total:.2f}%)")

## 5. Ergodic Distribution

In [ ]:
start = time.time()
eigenvalues, eigenvectors = sp_linalg.eigs(
    TranMatrix, k=1, which="LM", v0=np.ones(N_states)
)
ergodic_dist = eigenvectors[:, 0].real
ergodic_dist = ergodic_dist / ergodic_dist.sum()

elapsed = time.time() - start
print(f"Ergodic distribution computed in {elapsed:.2f} seconds")
print()
for j in range(J):
    mass_j = ergodic_dist[j * M : (j + 1) * M].sum()
    print(
        f"{state_names[j]:15s}: TM mass = {mass_j:.4f}, "
        f"stationary = {markov_stationary[j]:.4f}"
    )

In [ ]:
TM_C = sum(np.dot(cPol[j], ergodic_dist[j * M : (j + 1) * M]) for j in range(J))
TM_A = sum(np.dot(aPol[j], ergodic_dist[j * M : (j + 1) * M]) for j in range(J))

print(f"TM Aggregate Consumption = {TM_C:.6f}")
print(f"TM Aggregate Assets      = {TM_A:.6f}")

## 6. Comparison

In [ ]:
print("=== Aggregate Comparison ===")
print(f"{'':20s} {'MC':>12s} {'TM':>12s} {'Diff':>12s}")
print(f"{'Consumption':20s} {MC_C:12.6f} {TM_C:12.6f} {MC_C - TM_C:12.6f}")
print(f"{'Assets':20s} {MC_A:12.6f} {TM_A:12.6f} {MC_A - TM_A:12.6f}")

### Time series comparison

In [ ]:
dstn = ergodic_dist.copy()
tm_aLvls = []
for t in range(agent.T_sim - burn_in):
    A_val = sum(np.dot(aPol[j], dstn[j * M : (j + 1) * M]) for j in range(J))
    tm_aLvls.append(A_val)
    dstn = TranMatrix @ dstn

plt.figure(figsize=(16, 6))
plt.plot(mc_aLvls[burn_in:], label="Monte Carlo", alpha=0.7, linewidth=0.8)
plt.plot(tm_aLvls, label="Transition Matrix", linewidth=2.5)
plt.xlabel("Period")
plt.ylabel("Aggregate Assets (normalized)")
plt.title("MC vs TM: Aggregate Assets — Serial Unemployment Model")
plt.legend(fontsize=12)
plt.show()

### Distribution of $m$ by Markov state

With degenerate income (unemployed get exactly 0), the unemployed distribution
should show a sharp spike near $m = 0$ — all their market resources come from
prior savings times $R$.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for j in range(J):
    ax = axes[j // 2][j % 2]
    p_j = ergodic_dist[j * M : (j + 1) * M]
    mass_j = p_j.sum()
    p_j_cond = p_j / mass_j if mass_j > 0 else p_j

    ax.plot(dist_mGrid, p_j_cond, label="TM", linewidth=2)

    in_state_j = agent.shocks["Mrkv"] == j
    mc_m_j = agent.state_now["mNrm"][in_state_j]
    if len(mc_m_j) > 0:
        h, _ = np.histogram(mc_m_j, bins=dist_mGrid)
        h = h / h.sum()
        ax.plot(dist_mGrid[:-1], h, label="MC", linewidth=1.5, alpha=0.8)

    ax.set_title(f"{state_names[j]} (mass = {mass_j:.4f})", fontsize=12)
    ax.set_xlabel("$m$")
    ax.set_xlim([0, 20])
    ax.legend()

axes[0][0].set_ylabel("P(m | state)")
axes[1][0].set_ylabel("P(m | state)")
plt.suptitle("Distribution of $m$ by Markov State", fontsize=14)
plt.tight_layout()
plt.show()

### Grid convergence

In [ ]:
grid_sizes = [50, 100, 200, 300]
tm_assets_by_grid = []

for mC in grid_sizes:
    g = make_grid_exp_mult(ming=mMin, maxg=mMax, ng=mC, timestonest=mFac)
    M_g = len(g)
    N_g = M_g * J

    cP = [agent.solution[0].cFunc[j](g) for j in range(J)]
    aP = [np.maximum(g - cP[j], 0.0) for j in range(J)]

    NBD = np.zeros(N_g)
    for jp in range(J):
        sd = IncShkDstn_list[jp]
        nb_m = jump_to_grid_1D(sd.atoms[1], sd.pmv, g)
        NBD[jp * M_g : (jp + 1) * M_g] = MrkvPrbsInit[jp] * nb_m

    TM_g = np.zeros((N_g, N_g))
    for j in range(J):
        a_g = aP[j]
        LivPrb_j = LivPrb_arr[j]
        for jp in range(J):
            mp = MrkvArr[j, jp]  # row-stochastic
            if mp < 1e-15:
                continue
            Rfp = Rfree_arr[jp]
            PGFp = PermGroFac_arr[jp]
            sd = IncShkDstn_list[jp]
            bN = Rfp * a_g
            for i in range(M_g):
                mN = bN[i] / (sd.atoms[0] * PGFp) + sd.atoms[1]
                lw = jump_to_grid_1D(mN, sd.pmv, g)
                TM_g[jp * M_g : (jp + 1) * M_g, j * M_g + i] += mp * LivPrb_j * lw
        for i in range(M_g):
            TM_g[:, j * M_g + i] += (1.0 - LivPrb_j) * NBD

    ev, evec = sp_linalg.eigs(TM_g, k=1, which="LM", v0=np.ones(N_g))
    ed = evec[:, 0].real
    ed = ed / ed.sum()

    A_tm = sum(np.dot(aP[j], ed[j * M_g : (j + 1) * M_g]) for j in range(J))
    tm_assets_by_grid.append(A_tm)
    print(f"mCount={mC:4d}  TM Assets = {A_tm:.6f}")

print(f"MC Assets = {MC_A:.6f}")

In [ ]:
plt.figure(figsize=(10, 6))
for idx, mC in enumerate(grid_sizes):
    plt.axhline(
        y=tm_assets_by_grid[idx], linestyle="--", alpha=0.7, label=f"TM (mCount={mC})"
    )
plt.axhline(y=MC_A, color="black", linewidth=2, label="MC mean")
plt.ylabel("Aggregate Assets")
plt.title("Grid Convergence: Serial Unemployment Model")
plt.legend()
plt.show()

## 7. Summary

The same transition matrix code from the 2-state prototype works for the
4-state serial unemployment model with **no structural changes** — only
the parameters differ.  Key observations:

1. **Sparsity:** With degenerate income (1 shock point per state), each column
   of the transition matrix has at most $2J$ non-zero entries (2 from the lottery
   × $J$ possible target states).  This makes the TM very sparse (~0.5% non-zero).

2. **Unemployment distribution:** Unemployed agents accumulate at low $m$ values
   because they receive zero income ($\theta = 0$).  Their distribution is
   sharply peaked near $m = 0$, requiring a fine grid at the lower end.

3. **Markov state masses:** The TM ergodic distribution's marginal over Markov
   states matches the analytical stationary distribution of the 4×4 chain.

4. **Generality of the TM code:** The loop structure
   `for j in range(J): for jp in range(J): ...` handles any number of Markov
   states and any state-dependent parameters without modification.